In [1]:
from dotenv import load_dotenv
import getpass
import os

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your token:")

load_dotenv()

True

In [ ]:
import importlib
import eval_utils

importlib.reload(eval_utils)

In [2]:
from eval_utils import load_llm, load_embeddings, load_docs


# load chat model
chat_model = load_llm("hf_models/Llama-3.1-8B-Instruct")

# load embedding model
embedding = load_embeddings(model_id="sentence-transformers/all-MiniLM-L6-v2",
                            cache_dir="hf_models/embeddings/all-MiniLM-L6-v2")

# load docs
docs = load_docs("data/mk4s_manuel")

/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.40it/s]
Device set to use cuda:0
/home/a.zhang@PTW.Maschinenbau.TU-Darmstadt.de/project/RAGAS/eval_utils.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
100%|██████████| 8/8 [00:00<00:00, 9819.85it/s]


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
# chunking
size_lst =    [64, 64, 128, 128, 128, 256, 256, 256, 256]
overlap_lst = [16, 32, 16, 32, 64, 16, 32, 64, 128]

text_splitters = []

# chunking size list

for idx, (chunk_size, overlap) in enumerate(zip(size_lst, overlap_lst), start=1):
    var_name = f"text_splitter_{idx}_cs{chunk_size}_ov{overlap}"
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, 
                                                chunk_overlap=overlap,
                                                separators=["\n## ","\n### ","\n#### ", "\n\n", ".", "\n", " "],
                                                is_separator_regex=False,
                                                keep_separator=False,)
    text_splitters.append((var_name, text_splitter))

In [4]:
from langchain_community.vectorstores import FAISS
import time
import pandas as pd

# record parsing seconds
parsing_seconds = []
retrievers = []
names = []

# generate corresponding retriever
for var_name, splitter in text_splitters:

    t0 = time.perf_counter()
    chunks = splitter.split_documents(docs)
    vector_storage = FAISS.from_documents(chunks, embedding)
    t1 = time.perf_counter()

    parsing_seconds.append(t1-t0)
    names.append(var_name)

    retriever = vector_storage.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}
    )

    retrievers.append(retriever)

df_pars_time = pd.DataFrame({
    "splitter": names,
    "parsing_time": parsing_seconds
})


In [5]:
df_pars_time

,splitter,parsing_time
0,text_splitter_1_cs64_ov16,0.786789
1,text_splitter_2_cs64_ov32,0.312202
2,text_splitter_3_cs128_ov16,0.158790
3,text_splitter_4_cs128_ov32,0.161200
4,text_splitter_5_cs128_ov64,0.172908
5,text_splitter_6_cs256_ov16,0.102853
6,text_splitter_7_cs256_ov32,0.094463
7,text_splitter_8_cs256_ov64,0.092655
8,text_splitter_9_cs256_ov128,0.102628


In [ ]:
df_pars_time.to_csv('evaluate_results/03_chunk_test/parsing_time.csv')

In [ ]:
# build chain
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough,RunnableLambda
from langchain.schema.output_parser import StrOutputParser

template = """You are a helpful assistant specializing in 3D printing operations.  
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know. 
Use two sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:
"""

prompt = ChatPromptTemplate.from_template(template)
rag_chains = []

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

for retriever in retrievers:
    rag_chain = (
        {"context": retriever | RunnableLambda(format_docs),
         "question": RunnablePassthrough()}
        | prompt
        | chat_model
        | StrOutputParser()
    )
    rag_chains.append(rag_chain)

In [9]:
import pandas as pd
import time
from ragas import EvaluationDataset

def get_eval_ds(path, retriever, rag_chain):

    testset = pd.read_csv(path)
    queries = testset["user_input"].to_list()
    references = testset["reference"].to_list()

    # generate evaluation dataset
    rs_times = []
    dataset = []

    for query, reference in zip(queries, references):

        t0 = time.perf_counter()
        relevant_docs = [doc.page_content for doc in retriever.invoke(query)]
        response = rag_chain.invoke(query)
        t1 = time.perf_counter()
        
        dataset.append(
            {
                "user_input": query,
                "retrieved_contexts": relevant_docs,
                "response": response,
                "reference": reference,
            }
        )
        rs_times.append(t1 -t0)

    eval_ds = EvaluationDataset.from_list(dataset)

    return eval_ds, rs_times

In [10]:
from ragas import evaluate
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, AnswerAccuracy, AnswerRelevancy, Faithfulness, FactualCorrectness
from eval_utils import load_eval_model

def get_eval_result(eval_ds, metrics):
    evaluator_llm = load_eval_model()

    result = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        )
    
    return result

In [11]:
# select metrics
metrics=[LLMContextPrecisionWithReference(),
         LLMContextRecall(), 
         AnswerAccuracy(),
         Faithfulness(),]

In [12]:
data_path = "evaluate_dataset/test_dataset.csv"

results_lst = []
for retriever, rag_chain in zip(retrievers,rag_chains):
    eval_ds, response_time = get_eval_ds(data_path, retriever, rag_chain)
    result = get_eval_result(eval_ds, metrics)
    results_lst.append((result,response_time))

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Evaluating: 100%|██████████| 160/160 [04:45<00:00,  1.78s/it]


In [ ]:
for idx, (result, response_time) in enumerate(results_lst):
    print(names[idx], result)
    df = result.to_pandas()
    df["response_time"] = response_time
    df.to_csv(f"./evaluate_results/03_chunk_test/{names[idx]}.csv")

text_splitter_1_cs64_ov16 {'llm_context_precision_with_reference': 0.2714, 'context_recall': 0.2427, 'nv_accuracy': 0.3750, 'faithfulness': 0.2850}
text_splitter_2_cs64_ov32 {'llm_context_precision_with_reference': 0.2539, 'context_recall': 0.2392, 'nv_accuracy': 0.3563, 'faithfulness': 0.2499}
text_splitter_3_cs128_ov16 {'llm_context_precision_with_reference': 0.4549, 'context_recall': 0.3169, 'nv_accuracy': 0.3937, 'faithfulness': 0.5513}
text_splitter_4_cs128_ov32 {'llm_context_precision_with_reference': 0.4434, 'context_recall': 0.3392, 'nv_accuracy': 0.4188, 'faithfulness': 0.5852}
text_splitter_5_cs128_ov64 {'llm_context_precision_with_reference': 0.4867, 'context_recall': 0.3857, 'nv_accuracy': 0.4250, 'faithfulness': 0.4998}
text_splitter_6_cs256_ov16 {'llm_context_precision_with_reference': 0.6035, 'context_recall': 0.4365, 'nv_accuracy': 0.4250, 'faithfulness': 0.6462}
text_splitter_7_cs256_ov32 {'llm_context_precision_with_reference': 0.6087, 'context_recall': 0.4484, 'nv_ac